# Model-selection uncertainty & model averaging (Axis 3)

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only; no individual prognosis, no therapy ranking. Executed in CI (nbmake) so the result cannot silently break.

The virtual-trial divergence view shows *that* the eligible TGI models disagree. This notebook is its inferential completion: it splits the composed OS forecast's uncertainty into **within-model** (parameter/IIV noise, shrinkable by a bigger trial) and **between-model** (model-selection risk, *irreducible*) via the law of total variance, and combines the models into an honestly-weighted central forecast whose disagreement travels with it (`onkos.combine`).

Weights are **forecast-combination weights (Bates–Granger), not posterior model probabilities** — the models are fit to different trials, so a posterior model probability is not identifiable.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The decomposition

`model_selection_fraction = BETWEEN / (WITHIN + BETWEEN)` — the share of forecast uncertainty that more data on any *one* model cannot remove.

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 156, 313)
cmp = onkos.compare(ds, purpose='tgi', context=ctx, drug_effect=1.0, t=t)
ma = cmp.model_average(target='median_os_weeks', endpoint='OS', weights='equal', n=200)

print('included:', [tr.record_id for tr in cmp.included])
print(f'model-averaged median OS = {ma.point:.1f} weeks  [tier {ma.tier}]')
print(f'within (parameter)      = {ma.within_var:.1f}')
print(f'between (model-choice)  = {ma.between_var:.1f}')
print(f'model_selection_fraction = {ma.model_selection_fraction:.2f}')
assert 0.0 <= ma.model_selection_fraction <= 1.0

## Weight-scheme sensitivity

The headline target is reported under every applicable scheme. A small swing means the average is robust; a large one is flagged. Weight choice is itself an uncertainty.

In [ ]:
dec = cmp.uncertainty_decomposition(target='median_os_weeks', n=200)
print(f"{'scheme':<10}{'point':>8}{'within':>9}{'between':>9}{'frac':>7}")
for scheme, row in dec.items():
    print(f"{scheme:<10}{row['point']:>8.1f}{row['within_var']:>9.1f}"
          f"{row['between_var']:>9.1f}{row['model_selection_fraction']:>7.2f}")
print(f'\nweight_sensitivity (point swing across schemes) = {ma.weight_sensitivity:.2f}')

## The model-averaged OS curve with its between-model band

A convex combination of monotone survival curves is itself a valid survival function; the band is the pointwise between-model ±1σ.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for tr in cmp.included:
    ax.plot(t, tr.os_curve, lw=1.0, alpha=0.7, label=f'{tr.record_id.split(".")[1]} [{tr.tier}]')
band = ma.between_band
ax.fill_between(t, np.clip(ma.curve - band, 0, 1), np.clip(ma.curve + band, 0, 1),
                color='black', alpha=0.12, label='between-model ±1σ')
ax.plot(t, ma.curve, color='black', lw=2.2, label=f'model average [{ma.tier}]')
ax.axhline(0.5, ls=':', color='grey')
ax.set(title=f'Model-averaged OS — NSCLC 1L (fraction {ma.model_selection_fraction:.0%})',
       xlabel='weeks', ylabel='survival fraction', ylim=(0, 1.02))
ax.legend(fontsize=7)
fig.tight_layout()

# Survival-function validity (research spec §6 landmark)
assert np.isclose(ma.curve[0], 1.0, atol=1e-6)
assert np.all(np.diff(ma.curve) <= 1e-9)

## Guardrails: averaging cannot manufacture confidence

Worst-input-wins still governs; a single-model context yields fraction 0 *and* a warning (a zero is an absence of cross-checks, not a clean bill of health); the clinical-use prohibition and the fraction are inseparable from the point estimate.

In [ ]:
# Averaged tier is the worst included tier — averaging cannot raise it.
assert ma.tier == max(tr.tier for tr in cmp.included)

# Degenerate set (M=1): fraction 0 + single_eligible_model warning.
one = onkos.compare(ds, purpose='tgi', context=ctx, drug_effect=1.0, t=t)
one.included = one.included[:1]
ma1 = one.model_average(n=80)
print('M=1 fraction:', ma1.model_selection_fraction, '| warnings:', ma1.warnings)
assert ma1.model_selection_fraction == 0.0
assert any('single_eligible_model' in w for w in ma1.warnings)

# The result carries the prohibition and never drops its disagreement.
d = ma.to_dict()
assert d['NOT_FOR_CLINICAL_USE'] is True and 'PROHIBITED' in d['onkos:clinicalUse']
assert d['onkos:modelSelectionUncertainty']['fraction'] == ma.model_selection_fraction
print('weights are', d['weights_meaning'])

## Curation triage: rank contexts by irreducible model-choice risk

High between-model fraction = the context where adding a better-validated TGI model has the most value. This is the model-level analog of the parameter-level sensitivity triage.

In [ ]:
for tt, ln in [('NSCLC','first'),('NSCLC','second'),('breast','first'),('CRC','first'),('HCC','first'),('melanoma','first')]:
    c = onkos.compare(ds, purpose='tgi', context={'tumor_type': tt, 'line': ln}, drug_effect=1.0, t=t)
    if len(c.included) < 2:
        continue
    m = c.model_average(n=120)
    print(f'{tt:9} {ln:7} M={len(c.included)}  model-selection fraction {m.model_selection_fraction:.0%}')